# Cross-Viewpoint Protocol — Real-IAD Robustness Benchmark

Evaluates model robustness to unseen camera viewpoints.
Models are trained on viewpoints C1 and C2 only, then evaluated on
the unseen viewpoints C3, C4, and C5.

This protocol is novel for all three models and directly addresses
the research question on viewpoint robustness in industrial inspection.

Models: AnomalyDINO, Dinomaly, INP-Former
Dataset: Real-IAD 512px, 30 categories
Reference: Standard protocol results from 02_standard_protocol.ipynb

In [ ]:
# Reduce CUDA memory fragmentation
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

# Clone repo if not already present, otherwise pull latest
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

# Force INP-Former submodule to correct commit with path fixes
!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

# Install dependencies
!pip install anomalib==2.3.3 ADEval einops timm kornia -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import shutil

local_dataset_root = '/content/realiad_512'

if not os.path.exists(local_dataset_root):
    print("Copying Real-IAD from Drive to local storage...")
    shutil.copytree(dataset_root, local_dataset_root)
    print(f"Dataset ready at: {local_dataset_root}")
else:
    print(f"Dataset already on local storage: {local_dataset_root}")

dataset_root = local_dataset_root
print(f"Active dataset root: {dataset_root}")

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
import gc

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_all = realiad_utils.load_realiad_all
get_crossview_split = realiad_utils.get_crossview_split
train_dinomaly = trainer.train_dinomaly
train_anomalydino = trainer.train_anomalydino
train_inpformer = trainer.train_inpformer
run_inference = trainer.run_inference
run_inference_inpformer = trainer.run_inference_inpformer
compute_i_auroc = metrics.compute_i_auroc
compute_s_auroc = metrics.compute_s_auroc
compute_degradation_ratio = metrics.compute_degradation_ratio

print("All modules loaded")

## Step 1: Load Real-IAD and Apply Cross-Viewpoint Split

Training is restricted to viewpoints C1 and C2.
Evaluation is performed on viewpoints C3, C4, and C5 only.
This simulates a deployment scenario where the model is commissioned
on a limited set of camera angles and must generalise to new viewpoints.

In [ ]:
# Load all 30 categories
df = load_realiad_all(data_root=dataset_root)

# Apply cross-viewpoint split
train_views = ['C1', 'C2']
test_views = ['C3', 'C4', 'C5']

train_df_cv, test_df_cv = get_crossview_split(
    df,
    train_views=train_views,
    test_views=test_views
)

# Training set: normal images from C1 and C2 only
train_df_cv = train_df_cv[train_df_cv['label'] == 0].reset_index(drop=True)

print(f"Train views: {train_views}")
print(f"Test views: {test_views}")
print(f"Train (normal only): {len(train_df_cv)}")
print(f"Test total: {len(test_df_cv)}")
print(f"Test label distribution:\n{test_df_cv['label'].value_counts()}")

os.makedirs(f'{repo_path}/results', exist_ok=True)
df.to_csv(f'{repo_path}/results/dataset_split_crossview.csv', index=False)
print("Dataset split saved")

In [ ]:
## Step 2: Dinomaly — Cross-Viewpoint Protocol

Same architecture and training configuration as standard protocol.
Only the training data differs: C1+C2 viewpoints instead of all five.

In [ ]:
model_dinomaly_cv = train_dinomaly(
    train_df=train_df_cv,
    n_iterations=50000,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/dinomaly_crossview.pth'
)

results_dinomaly_cv = run_inference(
    model=model_dinomaly_cv,
    test_df=test_df_cv,
    model_name='Dinomaly',
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_dinomaly_cv.to_csv(
    f'{repo_path}/results/dinomaly_crossview_scores.csv', index=False)

i_auroc_din_cv = compute_i_auroc(results_dinomaly_cv)
s_auroc_din_cv = compute_s_auroc(results_dinomaly_cv)
print(f"I-AUROC Dinomaly (cross-view): {i_auroc_din_cv:.4f}")
print(f"S-AUROC Dinomaly (cross-view): {s_auroc_din_cv:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_dinomaly_cv
print("GPU memory cleared")

## Step 3: AnomalyDINO — Cross-Viewpoint Protocol

Memory bank constructed from C1+C2 normal features only.
The nearest-neighbour search at inference time uses only these two-viewpoint features.

In [ ]:
model_anomalydino_cv = train_anomalydino(
    train_df=train_df_cv,
    device='cuda',
    repo_path=repo_path,
    sampling_ratio=0.1,
    save_path=f'{repo_path}/results/weights/anomalydino_crossview.pt'
)

results_anomalydino_cv = run_inference(
    model=model_anomalydino_cv,
    test_df=test_df_cv,
    model_name='AnomalyDINO',
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_anomalydino_cv.to_csv(
    f'{repo_path}/results/anomalydino_crossview_scores.csv', index=False)

i_auroc_dino_cv = compute_i_auroc(results_anomalydino_cv)
s_auroc_dino_cv = compute_s_auroc(results_anomalydino_cv)
print(f"I-AUROC AnomalyDINO (cross-view): {i_auroc_dino_cv:.4f}")
print(f"S-AUROC AnomalyDINO (cross-view): {s_auroc_dino_cv:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_anomalydino_cv
print("GPU memory cleared")

## Step 4: INP-Former — Cross-Viewpoint Protocol

Prototype tokens are learned from C1+C2 normal features only.
At inference time the prototypes must generalise to the unseen C3+C4+C5 viewpoints.

In [ ]:
model_inpformer_cv = train_inpformer(
    train_df=train_df_cv,
    dataset_root=dataset_root,
    n_epochs=200,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_crossview.pth'
)

results_inpformer_cv = run_inference_inpformer(
    model=model_inpformer_cv,
    test_df=test_df_cv,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_inpformer_cv.to_csv(
    f'{repo_path}/results/inpformer_crossview_scores.csv', index=False)

i_auroc_inp_cv = compute_i_auroc(results_inpformer_cv)
s_auroc_inp_cv = compute_s_auroc(results_inpformer_cv)
print(f"I-AUROC INP-Former (cross-view): {i_auroc_inp_cv:.4f}")
print(f"S-AUROC INP-Former (cross-view): {s_auroc_inp_cv:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inpformer_cv
print("GPU memory cleared")

## Step 5: Performance Degradation Analysis

Compares cross-viewpoint results against standard protocol results.
The degradation ratio quantifies sensitivity to viewpoint shift.
A higher ratio indicates greater dependence on training viewpoint coverage.

In [ ]:
# Load standard protocol results for comparison
results_dinomaly_std = pd.read_csv(
    f'{repo_path}/results/dinomaly_standard_scores.csv')
results_anomalydino_std = pd.read_csv(
    f'{repo_path}/results/anomalydino_standard_scores.csv')
results_inpformer_std = pd.read_csv(
    f'{repo_path}/results/inpformer_standard_scores.csv')

# Compute standard protocol I-AUROC values
i_auroc_din_std = compute_i_auroc(results_dinomaly_std)
i_auroc_dino_std = compute_i_auroc(results_anomalydino_std)
i_auroc_inp_std = compute_i_auroc(results_inpformer_std)

# Compute degradation ratios
deg_dinomaly = compute_degradation_ratio(i_auroc_din_std, i_auroc_din_cv)
deg_anomalydino = compute_degradation_ratio(i_auroc_dino_std, i_auroc_dino_cv)
deg_inpformer = compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_cv)

# Results table
summary = pd.DataFrame({
    'Model': ['Dinomaly', 'AnomalyDINO', 'INP-Former'],
    'I-AUROC Standard': [i_auroc_din_std, i_auroc_dino_std, i_auroc_inp_std],
    'I-AUROC Cross-View': [i_auroc_din_cv, i_auroc_dino_cv, i_auroc_inp_cv],
    'Degradation (%)': [deg_dinomaly, deg_anomalydino, deg_inpformer],
})

print("=" * 60)
print("CROSS-VIEWPOINT PROTOCOL RESULTS")
print("=" * 60)
print(summary.round(4).to_string(index=False))

summary.to_csv(f'{repo_path}/results/summary_crossview.csv', index=False)
print("\nResults saved to results/summary_crossview.csv")

# WGA on cross-view results
wga_module = load_module("wga", f"{repo_path}/evaluation/wga.py")
print_wga_summary = wga_module.print_wga_summary

df_dict_cv = {
    'Dinomaly': results_dinomaly_cv,
    'AnomalyDINO': results_anomalydino_cv,
    'INP-Former': results_inpformer_cv
}

print_wga_summary(df_dict_cv)